# W1 Fair Rematch — RF-DETR vs YOLOv11-seg on the **21cm scc21** set

Derived from `rfdetr_pipeline.ipynb`. The evaluator, SAM, and product-export cells are carried over
unchanged; what differs is the data and the fairness of the YOLO arm.

### What changed vs the 60cm notebook
1. **Data:** `scc21_colab.zip` — 628 chips at 640px cropped from 21cm county imagery at **native
   0.21 m/px** (never resized). Built by `scripts/data/import_21cm_from_roboflow.py` +
   `pack_scc21_for_colab.py`. train=332 / val=100 / test=196.
2. **A fresh YOLO arm.** `R2-cameron` is a *60cm-trained* checkpoint — scoring it on 21cm chips
   measures domain shift, not architecture, and comparing it against an RF-DETR trained *on* these
   chips would corrupt Gate A. So YOLOv11-seg is retrained here on the same chips, and **it** is the
   parity target. R2 stays only as a reference line.
3. **Matched resolution.** RF-DETR runs at 728 (multiple of 56); YOLO at **736** (nearest multiple
   of 32). Neither model gets a scale advantage. Median object is ~18px at that input.

### Known caveats — read before trusting Gate A
- **Train is incomplete:** 83 of 175 source tiles (Akshitha's 88 were still unlabeled at build
  time). Both arms see identical data so the *relative* comparison is valid, but hold the Gate A
  verdict until the full train set lands, then re-run.
- 93–95% of objects are COCO-*small* and ~50% are under 16px. If both arms look recall-limited,
  run the YOLO arm again at `imgsz=1280` (cell at the end) before blaming architecture —
  `docs/PERMISSIVE_STACK_MIGRATION.md` guardrail 1 says resolution beats architecture here.

### Before you run
- Runtime → **GPU**.
- Upload `outputs/scc21_colab.zip` to `/content/drive/MyDrive/solar-soiling/`.


In [ ]:
# --- Runtime + Drive + config -------------------------------------------
import os, shutil, zipfile, json, datetime, torch
from pathlib import Path
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "!! CPU-only — set Runtime>GPU")

from google.colab import drive
drive.mount('/content/drive')

DRIVE            = '/content/drive/MyDrive/solar-soiling'   # same Drive as the ramp/bake-off notebooks
NAIP_ZIP         = f'{DRIVE}/scc21_colab.zip'               # 21cm chips, native 0.21 m/px
R2_WEIGHTS_DRIVE = f'{DRIVE}/models/r2_cameron_20260509.pt' # AGPL baseline, eval-only
WORK             = '/content/pipeline'
os.makedirs(WORK, exist_ok=True)

# Eval knobs (match production) ------------------------------------------
IOU_THRESH  = 0.50     # production iou
CONF_REPORT = 0.40     # production operating point (F1 also swept for a fair cross-model number)

# RF-DETR W1 fair-rematch knobs (docs/PERMISSIVE_STACK_MIGRATION.md §W1) --
RFDETR_RES    = 728    # input resolution; MUST be a multiple of 56 (default 560). Higher = small-panel recall.
RFDETR_EPOCHS = 100    # real budget; watch overfit at 332 chips — the convergence cell decides if it was enough.
RFDETR_OUT    = f'{WORK}/rfdetr_out'
assert RFDETR_RES % 56 == 0, 'RF-DETR resolution must be a multiple of 56'

# YOLO fair-arm knobs -----------------------------------------------------
YOLO_IMGSZ  = 736      # nearest multiple of 32 to RFDETR_RES — matched magnification
YOLO_EPOCHS = 100
assert YOLO_IMGSZ % 32 == 0, 'ultralytics imgsz must be a multiple of 32'
assert abs(YOLO_IMGSZ - RFDETR_RES) <= 8, 'arms must be within one stride of each other'

DATE = datetime.date.today().strftime('%Y%m%d')
assert os.path.isfile(NAIP_ZIP), f'scc21_colab.zip not found at {NAIP_ZIP} — upload it to Drive first'
assert os.path.isfile(R2_WEIGHTS_DRIVE), f'R2 weights not found at {R2_WEIGHTS_DRIVE}'
R2_WEIGHTS = f'{WORK}/r2_cameron_20260509.pt'
shutil.copy(R2_WEIGHTS_DRIVE, R2_WEIGHTS)
print('Drive OK — training/evaluating on the GT inside', NAIP_ZIP)

In [ ]:
# --- Extract scc21_colab.zip -> local YOLO layout (Roboflow 'valid' -> 'val') --
NAIP_DIR = f'{WORK}/naip'
shutil.rmtree(NAIP_DIR, ignore_errors=True)
tmp = Path(f'{WORK}/naip_extract'); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
print('Extracting scc21_colab.zip ...')
with zipfile.ZipFile(NAIP_ZIP) as z:
    z.extractall(str(tmp))

split_map = {'train': 'train', 'valid': 'val', 'test': 'test'}
found = {}
for root, dirs, _ in os.walk(str(tmp)):
    for d in dirs:
        if d in split_map and d not in found:
            found[d] = Path(root)/d
    if len(found) == 3:
        break
base = Path(NAIP_DIR)
for src_name, dst_name in split_map.items():
    src = found.get(src_name)
    if not src:
        continue
    for sub in ('images', 'labels'):
        s = src/sub; dst = base/sub/dst_name
        if s.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(str(s), str(dst))
shutil.rmtree(tmp, ignore_errors=True)

NAIP = Path(NAIP_DIR)
SPLITS = {s: (NAIP/'images'/s, NAIP/'labels'/s) for s in ['train', 'val', 'test']}
for s, (i, l) in SPLITS.items():
    n_img = len(list(i.glob('*.png')) + list(i.glob('*.jpg'))) if i.exists() else 0
    n_lbl = len(list(l.glob('*.txt'))) if l.exists() else 0
    print(f'{s:5s}  images={n_img:4d}  labels={n_lbl:4d}')
print('Expected: train=332 val=100 test=196 chips (21cm, 640px native crops).')
print('Background chips with no arrays are intentional negatives — do not filter them.')

In [ ]:
# --- Engine-agnostic evaluator (verbatim port of src/utils/det_match) ----
# The single source of eval truth: greedy best-first IoU matching, same as production.
import numpy as np
from PIL import Image

def _iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = max(0.0, ax2-ax1)*max(0.0, ay2-ay1) + max(0.0, bx2-bx1)*max(0.0, by2-by1) - inter
    return inter/union if union > 0 else 0.0

def match_predictions(preds, gts, iou_thresh=0.5):
    matched, tps, fps = set(), [], []
    for pi, pb in enumerate(preds):
        best_iou, best_gt = 0.0, None
        for gi, gb in enumerate(gts):
            if gi in matched:
                continue
            v = _iou(pb, gb)
            if v > best_iou:
                best_iou, best_gt = v, gi
        if best_gt is not None and best_iou >= iou_thresh:
            tps.append((pi, best_gt, best_iou)); matched.add(best_gt)
        else:
            fps.append(pi)
    fns = [i for i in range(len(gts)) if i not in matched]
    return tps, fps, fns

def load_gt_norm(label_path):
    if not Path(label_path).exists():
        return []
    out = []
    for line in Path(label_path).read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        try:
            c = np.array([float(p) for p in parts[1:]], dtype=np.float64)
        except ValueError:
            continue
        xs, ys = c[0::2], c[1::2]
        if len(xs) < 3:
            continue
        out.append((float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())))
    return out

def image_size(p):
    with Image.open(p) as im:
        return im.size

def _f1(tp, fp, fn):
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f

def evaluate(preds_by_img, gt_by_img):
    scored, n_gt = [], 0
    for stem, gts in gt_by_img.items():
        n_gt += len(gts)
        dets = sorted(preds_by_img.get(stem, []), key=lambda d: -d[0])
        _tp, _fp, _ = match_predictions([b for _, b in dets], gts, IOU_THRESH)
        tp_idx = {pi for pi, _, _ in _tp}
        for i, (sc, _) in enumerate(dets):
            scored.append((sc, 1 if i in tp_idx else 0))
    scored.sort(key=lambda x: -x[0])
    tp_c = fp_c = 0; rec, prec = [], []
    for _, is_tp in scored:
        tp_c += is_tp; fp_c += (1-is_tp)
        rec.append(tp_c/n_gt if n_gt else 0.0)
        prec.append(tp_c/(tp_c+fp_c))
    ap = 0.0
    for t in np.linspace(0, 1, 101):
        ps = [p for p, r in zip(prec, rec) if r >= t]
        ap += (max(ps) if ps else 0.0)/101
    best = (0.0, 0.0, 0.0, 0.0)
    at_report = (0.0, 0.0, 0.0)
    for conf in np.linspace(0.05, 0.95, 19):
        TP = FP = FN = 0
        for stem, gts in gt_by_img.items():
            dets = sorted([d for d in preds_by_img.get(stem, []) if d[0] >= conf], key=lambda d: -d[0])
            tp, fp, fn = match_predictions([b for _, b in dets], gts, IOU_THRESH)
            TP += len(tp); FP += len(fp); FN += len(fn)
        p, r, f = _f1(TP, FP, FN)
        if f > best[0]:
            best = (f, float(conf), p, r)
        if abs(conf - CONF_REPORT) < 0.026:
            at_report = (p, r, f)
    return {'ap50': round(ap, 3), 'best_f1': round(best[0], 3), 'best_conf': round(best[1], 2),
            'f1_at_report': round(at_report[2], 3), 'p_at_report': round(at_report[0], 3),
            'r_at_report': round(at_report[1], 3), 'n_gt': n_gt}

def split_paths(split):
    i, _ = SPLITS[split]
    return sorted(list(i.glob('*.png')) + list(i.glob('*.jpg')))

def gt_for(split):
    _, l = SPLITS[split]
    return {p.stem: load_gt_norm(l/(p.stem+'.txt')) for p in split_paths(split)}

def run_eval(predict_fn, splits=('val', 'test')):
    rows = {}
    for s in splits:
        paths = split_paths(s)
        rows[s] = evaluate(predict_fn(paths), gt_for(s))
        print(f'  [{s}] {rows[s]}')
    return rows

print('evaluator ready')

## 1a. R2-cameron baseline — re-measured on the *current* labels

The rematch is only fair if both detectors are scored on the same, current GT. Since you relabeled,
the old bake-off numbers (R2 test AP@50 0.616) are stale — so we re-score **frozen** R2 here on
whatever GT is in the uploaded zip. This is eval-only (AGPL); the weights are never shipped.

In [ ]:
# --- Baseline: R2-cameron, whole-tile (no SAHI), on current GT -----------
!pip -q install ultralytics
from ultralytics import YOLO
_r2 = YOLO(R2_WEIGHTS, task='segment')

def predict_r2(img_paths):
    out = {}
    for p in img_paths:
        r = _r2.predict(source=str(p), imgsz=640, conf=0.01, iou=0.7, verbose=False)[0]
        w, h = image_size(p)
        dets = []
        if r.boxes is not None and len(r.boxes) > 0:
            xyxy = r.boxes.xyxy.cpu().numpy(); conf = r.boxes.conf.cpu().numpy()
            for (x1, y1, x2, y2), c in zip(xyxy, conf):
                dets.append((float(c), (x1/w, y1/h, x2/w, y2/h)))
        out[p.stem] = dets
    return out

print('R2-cameron (whole-tile) re-measured on current GT -- the fair parity target:')
R2 = run_eval(predict_r2)

In [ ]:
# --- Fresh YOLOv11-seg on scc21 — the FAIR YOLO arm ----------------------
# R2-cameron above is 60cm-trained; on 21cm chips it measures domain shift, not architecture.
# This trains YOLO from its own base on the SAME chips RF-DETR gets, at matched resolution.
# Hyperparameters are the CLAUDE.md invariants (SGD/lr0/auto_augment/... must survive any edit).
from ultralytics import YOLO

Path(f'{WORK}/scc21.yaml').write_text(
    f"path: {NAIP_DIR}\ntrain: images/train\nval: images/val\ntest: images/test\n"
    f"nc: 1\nnames: ['solar_array']\n")

_yt = YOLO('yolo11s-seg.pt')
_yt.train(data=f'{WORK}/scc21.yaml', epochs=YOLO_EPOCHS, imgsz=YOLO_IMGSZ, batch=8,
          optimizer='SGD', lr0=0.001, auto_augment=None, erasing=0.0, translate=0.0,
          mosaic=0.5, copy_paste=0.0,
          project=f'{WORK}/yolo_out', name=f'scc21_{YOLO_IMGSZ}', exist_ok=True)

YOLO_BEST = f'{WORK}/yolo_out/scc21_{YOLO_IMGSZ}/weights/best.pt'
assert os.path.isfile(YOLO_BEST), f'no best.pt at {YOLO_BEST}'
_y21 = YOLO(YOLO_BEST, task='segment')

def predict_yolo21(img_paths):
    out = {}
    for p in img_paths:
        r = _y21.predict(source=str(p), imgsz=YOLO_IMGSZ, conf=0.01, iou=0.7, verbose=False)[0]
        w, h = image_size(p)
        dets = []
        if r.boxes is not None and len(r.boxes) > 0:
            xyxy = r.boxes.xyxy.cpu().numpy(); conf = r.boxes.conf.cpu().numpy()
            for (x1, y1, x2, y2), c in zip(xyxy, conf):
                dets.append((float(c), (x1/w, y1/h, x2/w, y2/h)))
        out[p.stem] = dets
    return out

print(f'YOLOv11-seg @{YOLO_IMGSZ}, trained on scc21 — the fair parity target:')
YOLO21 = run_eval(predict_yolo21)


## 1b. RF-DETR — the committed detector (W1 fair-rematch budget)

Apache-2.0, built for fast fine-tuning on small custom datasets, consumes COCO (same as Roboflow
exports). The 1st cold run (res 560 / 60 ep) lost to R2 — but unfairly, since R2 rode a domain
warm-start chain. This gives RF-DETR a real budget: **res 728**, **effective batch 16** (4×4 grad-accum),
**~100 epochs**, then the **convergence cell** confirms val AP plateaued before we read Gate A.

In [ ]:
# --- YOLO-seg -> COCO converter (RF-DETR / Roboflow layout) --------------
def yolo_to_coco(split, dst_dir, copy_images=True):
    img_dir, lbl_dir = SPLITS[split]
    os.makedirs(dst_dir, exist_ok=True)
    images, annos, ann_id = [], [], 1
    paths = sorted(list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg')))
    for iid, p in enumerate(paths, 1):
        w, h = image_size(p)
        images.append({'id': iid, 'file_name': p.name, 'width': w, 'height': h})
        if copy_images:
            shutil.copy(p, Path(dst_dir)/p.name)
        for (x1, y1, x2, y2) in load_gt_norm(lbl_dir/(p.stem+'.txt')):
            bx, by, bw, bh = x1*w, y1*h, (x2-x1)*w, (y2-y1)*h
            annos.append({'id': ann_id, 'image_id': iid, 'category_id': 1,
                          'bbox': [bx, by, bw, bh], 'area': bw*bh, 'iscrowd': 0})
            ann_id += 1
    coco = {'images': images, 'annotations': annos,
            'categories': [{'id': 1, 'name': 'solar_array', 'supercategory': 'none'}]}
    with open(Path(dst_dir)/'_annotations.coco.json', 'w') as f:
        json.dump(coco, f)
    return len(images), len(annos)

RFDIR = f'{WORK}/rfdetr_data'
for split, dst in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
    n_i, n_a = yolo_to_coco(split, f'{RFDIR}/{dst}')
    print(f'{dst:6s} images={n_i:4d} boxes={n_a:5d}')

In [ ]:
# --- RF-DETR train (W1) --------------------------------------------------
# Training deps live in the [train] extra; plain `pip install rfdetr` only pulls
# inference deps and .train() will ModuleNotFoundError.
!pip -q install "rfdetr[train,loggers]"
from rfdetr import RFDETRBase

shutil.rmtree(RFDETR_OUT, ignore_errors=True); os.makedirs(RFDETR_OUT, exist_ok=True)
rf = RFDETRBase(resolution=RFDETR_RES)
rf.train(dataset_dir=RFDIR, epochs=RFDETR_EPOCHS, batch_size=4,
         grad_accum_steps=4, lr=1e-4, output_dir=RFDETR_OUT)
print('training done ->', RFDETR_OUT)

In [ ]:
# --- Convergence check: did val AP plateau? (else it was undertrained) ---
# RF-DETR writes per-epoch metrics to log.txt (one JSON object per line). We pull
# whatever COCO-eval AP array it logged and plot AP@0.50 (index 1) over epochs.
import matplotlib.pyplot as plt
log_path = Path(RFDETR_OUT)/'log.txt'
epochs_ap = []
if log_path.exists():
    for line in log_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except Exception:
            continue
        ep = rec.get('epoch')
        ap_arr = next((rec[k] for k in rec if 'coco_eval_bbox' in k and isinstance(rec[k], (list, tuple))), None)
        if ep is not None and ap_arr and len(ap_arr) >= 2:
            epochs_ap.append((ep, float(ap_arr[1]), float(ap_arr[0])))  # (epoch, AP@0.50, AP@[.5:.95])

if epochs_ap:
    ep, ap50, apc = zip(*sorted(epochs_ap))
    plt.figure(figsize=(7, 4))
    plt.plot(ep, ap50, marker='o', label='val AP@0.50')
    plt.plot(ep, apc, marker='.', alpha=0.6, label='val AP@[.5:.95]')
    plt.xlabel('epoch'); plt.ylabel('AP'); plt.legend(); plt.grid(alpha=0.3)
    plt.title('RF-DETR validation AP per epoch'); plt.show()
    tail = ap50[-5:]
    peak = max(ap50); peak_ep = ep[ap50.index(peak)]
    slope = (tail[-1] - tail[0]) / max(1, (len(tail)-1))
    print(f'peak val AP@0.50 = {peak:.3f} at epoch {peak_ep} / {ep[-1]}')
    print(f'last-5-epoch slope = {slope:+.4f} per epoch')
    if peak_ep >= ep[-1] - 2 and slope > 0.002:
        print('VERDICT: still climbing at the end -> UNDERTRAINED. Raise RFDETR_EPOCHS and re-run.')
    else:
        print('VERDICT: plateaued -> budget was adequate. Gate A below is trustworthy.')
else:
    print('No parseable per-epoch AP in log.txt — inspect', log_path, 'manually before trusting Gate A.')

In [ ]:
# --- RF-DETR eval: score the BEST checkpoint (peak val AP), not last epoch
_best = next((Path(RFDETR_OUT)/c for c in
              ('checkpoint_best_ema.pth', 'checkpoint_best_total.pth', 'checkpoint_best_regular.pth')
              if (Path(RFDETR_OUT)/c).exists()), None)
if _best is not None:
    print(f'eval: loading best checkpoint {_best.name}')
    try:
        rf = RFDETRBase(pretrain_weights=str(_best), resolution=RFDETR_RES)
    except Exception as e:
        print(f'  (could not reload best ckpt: {e} — using in-memory last-epoch weights)')
else:
    print('eval: no best checkpoint found — using in-memory last-epoch weights')

def predict_rfdetr(img_paths, threshold=0.01):
    out = {}
    for p in img_paths:
        img = Image.open(p).convert('RGB')
        det = rf.predict(img, threshold=threshold)  # supervision.Detections: xyxy px, confidence
        w, h = img.size
        out[p.stem] = [(float(c), (x1/w, y1/h, x2/w, y2/h))
                       for (x1, y1, x2, y2), c in zip(det.xyxy, det.confidence)]
    return out

print('RF-DETR:')
RF = run_eval(predict_rfdetr)

In [ ]:
# --- Comparison table + Gate A verdict (docs/PERMISSIVE_STACK_MIGRATION.md §4)
import pandas as pd
def rows_for(name, res):
    return [{'model': name, 'split': s,
             'AP@0.50': r['ap50'], 'best_F1': r['best_f1'], '@conf': r['best_conf'],
             f'F1@{CONF_REPORT}': r['f1_at_report'], 'P': r['p_at_report'], 'R': r['r_at_report'],
             'n_gt': r['n_gt']} for s, r in res.items()]

data = []
for name, var in [('R2-cameron 60cm (reference only)', 'R2'),
                  (f'YOLOv11-seg @{YOLO_IMGSZ} scc21 (parity target)', 'YOLO21'),
                  (f'RF-DETR @{RFDETR_RES} scc21', 'RF')]:
    if var in globals():
        data += rows_for(name, globals()[var])
df = pd.DataFrame(data)
print(df.to_string(index=False))

GATE_AP, GATE_F1 = 0.60, 0.63
print('\n' + '=' * 64 + '\nGATE A (test split) — permissive-detector parity check\n' + '=' * 64)
if 'RF' in globals() and 'test' in RF:
    rf_ap, rf_f1 = RF['test']['ap50'], RF['test']['best_f1']
    # Parity target = the fresh YOLO trained on THESE chips. R2 (60cm) is not a fair target.
    _tgt = YOLO21 if ('YOLO21' in globals() and 'test' in YOLO21) else None
    if _tgt is None:
        print('WARNING: no fresh-YOLO arm — falling back to the 60cm R2 checkpoint, which is\n'
              '         a domain-shifted target. Gate A is NOT trustworthy in this mode.')
        _tgt = R2 if ('R2' in globals() and 'test' in R2) else None
    r2_ap = _tgt['test']['ap50'] if _tgt else None
    r2_f1 = _tgt['test']['best_f1'] if _tgt else None
    print(f'RF-DETR test : AP@50 {rf_ap:.3f}  best-F1 {rf_f1:.3f}')
    if r2_ap is not None:
        print(f'parity target: AP@50 {r2_ap:.3f}  best-F1 {r2_f1:.3f}   (YOLO trained on scc21)')
    hits_abs = (rf_ap >= GATE_AP) and (rf_f1 >= GATE_F1)
    near_r2  = (r2_ap is not None) and (rf_ap >= r2_ap - 0.03) and (rf_f1 >= r2_f1 - 0.03)
    if hits_abs or near_r2:
        verdict = 'PASS — parity (within noise). Proceed W2->W6; port justified.'
    elif r2_ap is not None and (rf_ap >= r2_ap - 0.10) and (rf_f1 >= r2_f1 - 0.10):
        verdict = 'PARTIAL — closes most of the gap. Continue to W2/W3 (30cm data), then re-gate.'
    else:
        verdict = 'FAIL — still far behind after a fair run. Escalate to Cameron (§4).'
    print(f'abs bar (AP>=0.60 & F1>=0.63): {"yes" if hits_abs else "no"}   within-noise of R2: {"yes" if near_r2 else "no"}')
    print(f'VERDICT: {verdict}')
else:
    print('RF-DETR test results not present — run the train+eval cells first.')
print('\nNOTE: train was 83/175 source tiles at build time. Re-run once the full train set '
      'lands before treating this verdict as final.')

In [ ]:
# --- Resolution ablation: YOLO @1280 on the same chips -------------------
# 93-95% of objects here are COCO-small and ~50% are under 16px. At imgsz=736 the median object is
# ~18px = ~2 cells on YOLO's stride-8 P3 head. Feeding the SAME native chips at 1280 puts it near
# 32px without inventing pixels — it changes the object-size-to-stride ratio, which is the same
# mechanism SAHI exploits at inference. If this moves recall materially, the residual gap is SCALE,
# not architecture (PERMISSIVE_STACK_MIGRATION.md guardrail 1). Run after the table above.
ABLATE = False   # flip to True to spend the extra GPU time

if ABLATE:
    _ya = YOLO('yolo11s-seg.pt')
    _ya.train(data=f'{WORK}/scc21.yaml', epochs=YOLO_EPOCHS, imgsz=1280, batch=2,
              optimizer='SGD', lr0=0.001, auto_augment=None, erasing=0.0, translate=0.0,
              mosaic=0.5, copy_paste=0.0,
              project=f'{WORK}/yolo_out', name='scc21_1280', exist_ok=True)
    _y1280 = YOLO(f'{WORK}/yolo_out/scc21_1280/weights/best.pt', task='segment')

    def predict_yolo1280(img_paths):
        out = {}
        for p in img_paths:
            r = _y1280.predict(source=str(p), imgsz=1280, conf=0.01, iou=0.7, verbose=False)[0]
            w, h = image_size(p)
            dets = []
            if r.boxes is not None and len(r.boxes) > 0:
                xyxy = r.boxes.xyxy.cpu().numpy(); conf = r.boxes.conf.cpu().numpy()
                for (x1, y1, x2, y2), c in zip(xyxy, conf):
                    dets.append((float(c), (x1/w, y1/h, x2/w, y2/h)))
            out[p.stem] = dets
        return out

    print('YOLOv11-seg @1280 (resolution ablation):')
    YOLO1280 = run_eval(predict_yolo1280)
    for s in ('val', 'test'):
        if s in YOLO21 and s in YOLO1280:
            d_f1 = YOLO1280[s]['best_f1'] - YOLO21[s]['best_f1']
            d_r  = YOLO1280[s]['r_at_report'] - YOLO21[s]['r_at_report']
            print(f'  {s}: best-F1 {d_f1:+.3f}, recall@{CONF_REPORT} {d_r:+.3f} vs @{YOLO_IMGSZ}')
else:
    print('ABLATE=False — set True to run the 1280 resolution ablation.')


## 2. SAM area stage — box → mask → `minAreaRect` → m²

This is the product's area/$ differentiator. First we validate SAM's mask *ceiling* by prompting
with **ground-truth boxes** (isolates mask quality from detector error), then Section 3 wires it to
RF-DETR's *actual* boxes for the end-to-end export.

> SAM2 (`facebook/sam2-hiera-large`). **SAM3 drops into the same `set_image` / `predict(box=...)` role**
> if released + permissive — swap the two lines, nothing else changes.

In [ ]:
# --- SAM area-quality validation (GT-box prompted -> mask ceiling) -------
!pip -q install sam2
import cv2
from sam2.sam2_image_predictor import SAM2ImagePredictor
predictor = SAM2ImagePredictor.from_pretrained('facebook/sam2-hiera-large')  # <- SAM3 goes here

GSD_M      = 0.2043   # SCC 2025 21cm chips, GROUND m/px (not the 3857 pixel size).
                      # Was 0.6 (NAIP) — a stale constant that inflated every m2 by ~8.6x.
                      # Range across all 249 tiles is 0.2043-0.2044, so a constant is safe
                      # here; exact per-chip values live in tile_index_chips.json.
N_SAMPLE   = 60
EVAL_SPLIT = 'val'

def load_gt_polys_norm(label_path):
    polys = []
    if not Path(label_path).exists():
        return polys
    for line in Path(label_path).read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 7:
            continue
        c = np.array([float(p) for p in parts[1:]], dtype=np.float64)
        xs, ys = c[0::2], c[1::2]
        if len(xs) >= 3:
            polys.append(np.column_stack([xs, ys]))
    return polys

records, examples = [], []
_, lbl_dir = SPLITS[EVAL_SPLIT]
for p in split_paths(EVAL_SPLIT):
    if len(records) >= N_SAMPLE:
        break
    polys = load_gt_polys_norm(lbl_dir/(p.stem+'.txt'))
    if not polys:
        continue
    im = np.array(Image.open(p).convert('RGB')); H, W = im.shape[:2]
    predictor.set_image(im)
    for poly in polys:
        if len(records) >= N_SAMPLE:
            break
        pk = poly * np.array([W, H])
        box = np.array([pk[:,0].min(), pk[:,1].min(), pk[:,0].max(), pk[:,1].max()])
        with torch.inference_mode():
            masks, _, _ = predictor.predict(box=box, multimask_output=False)
        m = masks[0].astype(np.uint8)
        gt = np.zeros((H, W), np.uint8); cv2.fillPoly(gt, [pk.astype(np.int32)], 1)
        inter = int((m & gt).sum()); union = int((m | gt).sum())
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        a_rect = 0.0
        if cnts:
            (cw, ch) = cv2.minAreaRect(max(cnts, key=cv2.contourArea))[1]; a_rect = cw*ch
        records.append(dict(iou=inter/union if union else 0.0,
                            a_mask=float(m.sum()), a_gt=float(gt.sum()), a_rect=a_rect))
        if len(examples) < 4:
            examples.append((im, pk, m, box))

d = pd.DataFrame(records)
pe_mask = (d.a_mask - d.a_gt).abs()/d.a_gt*100
pe_rect = (d.a_rect - d.a_gt).abs()/d.a_gt*100
print(f'SAM2 area validation on {len(d)} GT arrays (GT-box prompted -> mask ceiling):')
print(f'  median mask<->GT IoU       : {d.iou.median():.3f}')
print(f'  median |area err| (raw mask): {pe_mask.median():.1f}%')
print(f'  median |area err| (minRect) : {pe_rect.median():.1f}%   <- the production recipe')
print(f'  @ GSD={GSD_M} m/px  median GT array = {d.a_gt.median()*GSD_M**2:.1f} m2')

In [ ]:
# --- SAM overlays: GT polygon (green) vs SAM mask (red) ------------------
fig, axes = plt.subplots(1, len(examples), figsize=(4*len(examples), 4))
if len(examples) == 1:
    axes = [axes]
for ax, (im, pk, m, box) in zip(axes, examples):
    x1, y1, x2, y2 = [int(v) for v in box]; pad = 20
    xa, ya = max(0, x1-pad), max(0, y1-pad); xb, yb = x2+pad, y2+pad
    crop = im[ya:yb, xa:xb].copy()
    ov = crop.copy(); ov[m[ya:yb, xa:xb] > 0] = [255, 0, 0]
    crop = (0.5*crop + 0.5*ov).astype(np.uint8)
    cv2.polylines(crop, [(pk - [xa, ya]).astype(np.int32)], True, (0, 255, 0), 2)
    ax.imshow(crop); ax.axis('off'); ax.set_title('GT=green  SAM=red')
plt.tight_layout(); plt.show()

## 3. Full permissive pipeline → product artifacts

The go-forward payload. For every test tile we run **image → RF-DETR box → SAM mask → `minAreaRect`
polygon**, then write the **exact YOLO-format `.txt` contract** that `scripts/detect/infer.py`
produces today (`0 x1 y1 … xn yn`, normalized, 6-decimal). Because `export_polygons_geojson.py` and
all of Stage 2 read *files*, not the model, this is the entire port surface (§W5) exercised in one cell.

We also compute per-array m² (the $ input), then push three artifacts to Drive:
the best RF-DETR checkpoint, the label pack, and a `manifest.json` — so it drops straight into the repo.

In [ ]:
# --- Full pipeline: RF-DETR box -> SAM mask -> polygon -> YOLO .txt ------
# Operating conf: use RF-DETR's best-F1 conf from the VAL sweep (never tuned on test).
OP_CONF = RF['val']['best_conf'] if 'RF' in globals() and 'val' in RF else 0.5
print(f'operating conf (from val best-F1 sweep) = {OP_CONF}')

PACK = Path(f'{WORK}/rfdetr_pipeline_out'); shutil.rmtree(PACK, ignore_errors=True)
LABELS = PACK/'labels'; LABELS.mkdir(parents=True)
area_records = []

def mask_to_norm_poly(mask, W, H):
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None, 0.0
    c = max(cnts, key=cv2.contourArea)
    eps = 0.01 * cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, eps, True).reshape(-1, 2)
    if len(approx) < 3:
        return None, 0.0
    (rw, rh) = cv2.minAreaRect(c)[1]
    norm = [(x / W, y / H) for x, y in approx]
    return norm, float(rw * rh)  # area in px^2 (minAreaRect)

for p in split_paths('test'):
    img = Image.open(p).convert('RGB'); W, H = img.size
    det = rf.predict(img, threshold=OP_CONF)
    predictor.set_image(np.array(img))
    lines = []
    for (x1, y1, x2, y2), conf in zip(det.xyxy, det.confidence):
        with torch.inference_mode():
            masks, _, _ = predictor.predict(box=np.array([x1, y1, x2, y2]), multimask_output=False)
        norm, a_px = mask_to_norm_poly(masks[0], W, H)
        if norm is None:
            continue
        pts = ' '.join(f'{v:.6f}' for xy in norm for v in xy)
        lines.append(f'0 {pts}')  # class 0 = solar_array, matching production labels
        area_records.append(dict(tile=p.stem, conf=float(conf), area_m2=a_px * (GSD_M ** 2)))
    (LABELS/(p.stem + '.txt')).write_text('\n'.join(lines))

ar = pd.DataFrame(area_records)
print(f'wrote {len(list(LABELS.glob("*.txt")))} label files, {len(ar)} arrays total')
if len(ar):
    print(f'median detected array area = {ar.area_m2.median():.1f} m2  (GSD={GSD_M} m/px)')
    print(ar.head().to_string(index=False))
print('\nThese .txt files are the exact contract export_polygons_geojson.py consumes — the whole port surface.')

In [ ]:
# --- Save artifacts to Drive (checkpoint + label pack + manifest) --------
# So the next step (repo-side W5 port) just pulls these down — no re-training.
OUT_DRIVE = Path(DRIVE)/'models'
OUT_DRIVE.mkdir(parents=True, exist_ok=True)

ckpt_name = None
if _best is not None and _best.exists():
    ckpt_name = f'rfdetr_w1_{DATE}.pth'
    shutil.copy(_best, OUT_DRIVE/ckpt_name)
    print('saved checkpoint  ->', OUT_DRIVE/ckpt_name)
else:
    print('WARN: no best checkpoint on disk to save')

# Zip the label pack (drop into runs/segment/<name>/labels/ on the repo side).
pack_zip = f'{WORK}/rfdetr_labels_{DATE}'
shutil.make_archive(pack_zip, 'zip', PACK)
shutil.copy(pack_zip + '.zip', OUT_DRIVE/f'rfdetr_labels_{DATE}.zip')
print('saved label pack  ->', OUT_DRIVE/f'rfdetr_labels_{DATE}.zip')

manifest = {
    'date': DATE,
    'detector': 'rf-detr',
    'resolution': RFDETR_RES,
    'epochs': RFDETR_EPOCHS,
    'operating_conf': OP_CONF,
    'iou_thresh': IOU_THRESH,
    'gsd_m': GSD_M,
    'checkpoint': ckpt_name,
    'yolo_imgsz': YOLO_IMGSZ,
    'yolo_epochs': YOLO_EPOCHS,
    'rf_metrics': RF if 'RF' in globals() else None,
    # The parity target Gate A is actually judged against — omitting it made the
    # first run's artifact unreadable without the notebook output beside it.
    'yolo_scc21_metrics': YOLO21 if 'YOLO21' in globals() else None,
    'yolo_1280_ablation_metrics': YOLO1280 if 'YOLO1280' in globals() else None,
    'r2_baseline_metrics': R2 if 'R2' in globals() else None,
    'train_tiles_note': 'check import_qa.json: train may be a subset if seeded tiles were unreviewed',
    'note': 'W1 fair rematch; labels emit the infer.py .txt contract. Eval on current relabeled GT.',
}
if manifest['yolo_scc21_metrics'] is None:
    print('WARN: no fresh-YOLO arm in this manifest — Gate A cannot be judged from this file alone.')
with open(OUT_DRIVE/f'rfdetr_w1_{DATE}_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print('saved manifest    ->', OUT_DRIVE/f'rfdetr_w1_{DATE}_manifest.json')
print(json.dumps(manifest, indent=2))

## Next steps (repo side — W5 port)

Once Gate A **PASSES** and the artifacts are on Drive:
1. Pull `rfdetr_w1_<date>.pth` into `models/`, register it in `models/registry.yaml` (mark R2 eval-only).
2. Write `scripts/detect/rfdetr_infer.py` — mirror **this notebook's Section 3** as the product infer:
   image → RF-DETR box → SAM mask → normalized-polygon `.txt` in `runs/segment/<name>/labels/`.
3. Point `src/solarsoiled/cli.py::detect` at the new infer; leave `export_polygons_geojson.py` +
   Stage 2 **untouched** (they read the files this notebook already produces).
4. Decide the **SAHI** question from recall numbers (§W5): SAM-adapter vs manual tiling vs skip-if-30cm-suffices.
5. `verify` skill: drive `solarsoiled detect` end-to-end on an AOI, confirm `arrays.geojson` from the permissive path.

If Gate A is **PARTIAL/FAIL**, do **not** port — go to W2 (30cm dual-GSD build) and re-gate, or escalate (§4).